# 02 — Silver Layer: ETL Transformations

This notebook walks through the **Silver layer** ETL pipeline.

**Goal:** read raw Bronze data, apply cleansing and enrichment, validate quality,
and write clean Parquet to the Silver store.

**Silver contracts:**
- All critical columns are non-null.
- All prices are strictly positive.
- No duplicate `(ticker, trade_date)` pairs.
- `daily_return` = (close_t / close_{t-1}) − 1, per ticker.

In [11]:
import sys
sys.path.insert(0, "..")

import polars as pl
pl.Config.set_tbl_rows(20)

polars.config.Config

## 1. Read Bronze data

In [12]:
from d_processing.bronze.ingest import read_bronze

bronze_df = read_bronze(source="yahoo_finance")
print(f"Bronze rows: {len(bronze_df):,}  |  tickers: {bronze_df['ticker'].n_unique() if not bronze_df.is_empty() else 0}")
bronze_df.head(5)

{"timestamp": "2026-06-25T17:01:32.492128+00:00", "level": "INFO", "logger": "d_processing.bronze.ingest", "message": "Bronze read complete", "module": "ingest", "func": "read_bronze", "line": 107, "name": "d_processing.bronze.ingest", "msg": "Bronze read complete", "args": [], "levelname": "INFO", "levelno": 20, "pathname": "\\\\wsl.localhost\\ubuntu-llm\\home\\efcardoso\\projects\\b3-data-plataform\\i_notebooks\\..\\d_processing\\bronze\\ingest.py", "filename": "ingest.py", "exc_info": null, "exc_text": null, "stack_info": null, "lineno": 107, "funcName": "read_bronze", "created": 1782406892.4920447, "msecs": 492.0, "relativeCreated": 688803.1536, "thread": 8944, "threadName": "MainThread", "processName": "MainProcess", "process": 14144, "taskName": "Task-238", "rows": 720, "source": "yahoo_finance"}
Bronze rows: 720  |  tickers: 12


ticker,trade_date,open,high,low,close,adj_close,volume,source,ingested_at
str,date,f64,f64,f64,f64,f64,i64,str,"datetime[μs, UTC]"
"""PETR4.SA""",2026-03-27,48.5,49.439999,48.130001,49.41,47.916618,54170900,"""yahoo_finance""",2026-06-25 16:02:59.146098 UTC
"""VALE3.SA""",2026-03-27,78.470001,79.919998,78.169998,79.0,79.0,11101700,"""yahoo_finance""",2026-06-25 16:02:59.146098 UTC
"""ITUB4.SA""",2026-03-27,41.810001,41.810001,41.18,41.450001,41.026432,27910700,"""yahoo_finance""",2026-06-25 16:02:59.146098 UTC
"""BBDC4.SA""",2026-03-27,18.709999,18.809999,18.440001,18.52,18.1793,20727200,"""yahoo_finance""",2026-06-25 16:02:59.146098 UTC
"""ABEV3.SA""",2026-03-27,14.79,15.0,14.74,14.76,14.719015,17884700,"""yahoo_finance""",2026-06-25 16:02:59.146098 UTC


## 2. Apply each transformation step individually

In [13]:
from d_processing.silver.transform import (
    cast_types,
    remove_nulls,
    remove_invalid_prices,
    deduplicate,
    calculate_daily_return,
)

step1 = cast_types(bronze_df)
print("After cast_types:")
print(step1.schema)
step1.head(3)

After cast_types:
Schema({'ticker': String, 'trade_date': Date, 'open': Float64, 'high': Float64, 'low': Float64, 'close': Float64, 'adj_close': Float64, 'volume': Int64, 'source': String, 'ingested_at': Datetime(time_unit='us', time_zone='UTC'), 'open_price': Float64, 'high_price': Float64, 'low_price': Float64, 'close_price': Float64})


ticker,trade_date,open,high,low,close,adj_close,volume,source,ingested_at,open_price,high_price,low_price,close_price
str,date,f64,f64,f64,f64,f64,i64,str,"datetime[μs, UTC]",f64,f64,f64,f64
"""PETR4.SA""",2026-03-27,48.5,49.439999,48.130001,49.41,47.916618,54170900,"""yahoo_finance""",2026-06-25 16:02:59.146098 UTC,48.5,49.439999,48.130001,49.41
"""VALE3.SA""",2026-03-27,78.470001,79.919998,78.169998,79.0,79.0,11101700,"""yahoo_finance""",2026-06-25 16:02:59.146098 UTC,78.470001,79.919998,78.169998,79.0
"""ITUB4.SA""",2026-03-27,41.810001,41.810001,41.18,41.450001,41.026432,27910700,"""yahoo_finance""",2026-06-25 16:02:59.146098 UTC,41.810001,41.810001,41.18,41.450001


In [14]:
step2 = remove_nulls(step1)
step3 = remove_invalid_prices(step2)
step4 = deduplicate(step3)
step5 = calculate_daily_return(step4)

print(f"Rows after each step:")
print(f"  Bronze input  : {len(bronze_df):>7,}")
print(f"  cast_types    : {len(step1):>7,}")
print(f"  remove_nulls  : {len(step2):>7,}")
print(f"  invalid prices: {len(step3):>7,}")
print(f"  deduplicate   : {len(step4):>7,}")
print(f"  daily_return  : {len(step5):>7,}")

Rows after each step:
  Bronze input  :     720
  cast_types    :     720
  remove_nulls  :     720
  invalid prices:     720
  deduplicate   :     720
  daily_return  :     720


In [15]:
step5.select(["ticker", "trade_date", "close_price", "daily_return"]).head(15)

ticker,trade_date,close_price,daily_return
str,date,f64,f64
"""VALE3.SA""",2026-04-27,85.5,null
"""ABEV3.SA""",2026-03-27,14.76,-0.757556
"""VALE3.SA""",2026-04-28,84.389999,3.065029
"""RENT3.SA""",2026-04-23,49.389999,0.118433
"""WEGE3.SA""",2026-06-03,41.779999,1.003837
"""MGLU3.SA""",2026-05-04,8.0,-0.468085
"""BBDC4.SA""",2026-04-30,19.32,1.18552
"""PETR4.SA""",2026-04-29,48.959999,0.2
"""VALE3.SA""",2026-05-18,81.830002,0.897287


## 3. Run quality checks

In [16]:
from e_validation.quality_checks import run_silver_quality_checks

validated = run_silver_quality_checks(step5)
print("✔  All quality checks passed")

{"timestamp": "2026-06-25T17:01:49.917807+00:00", "level": "INFO", "logger": "e_validation.quality_checks", "message": "Running Silver quality checks", "module": "quality_checks", "func": "run_silver_quality_checks", "line": 48, "name": "e_validation.quality_checks", "msg": "Running Silver quality checks", "args": [], "levelname": "INFO", "levelno": 20, "pathname": "\\\\wsl.localhost\\ubuntu-llm\\home\\efcardoso\\projects\\b3-data-plataform\\i_notebooks\\..\\e_validation\\quality_checks.py", "filename": "quality_checks.py", "exc_info": null, "exc_text": null, "stack_info": null, "lineno": 48, "funcName": "run_silver_quality_checks", "created": 1782406909.9177618, "msecs": 917.0, "relativeCreated": 706228.8708, "thread": 8944, "threadName": "MainThread", "processName": "MainProcess", "process": 14144, "taskName": "Task-289", "rows": 720}
{"timestamp": "2026-06-25T17:01:49.920906+00:00", "level": "INFO", "logger": "e_validation.quality_checks", "message": "All Silver quality checks passe

## 4. Write to Silver and verify

In [17]:
from d_processing.silver.transform import write_silver, read_silver
from d_processing.silver.transform import transform_silver

silver_clean = transform_silver(bronze_df)
write_silver(silver_clean)

silver_df = read_silver()
print(f"Silver rows: {len(silver_df):,}")
silver_df.head(10)

{"timestamp": "2026-06-25T17:01:59.379590+00:00", "level": "INFO", "logger": "d_processing.silver.transform", "message": "Silver write complete", "module": "transform", "func": "write_silver", "line": 159, "name": "d_processing.silver.transform", "msg": "Silver write complete", "args": [], "levelname": "INFO", "levelno": 20, "pathname": "\\\\wsl.localhost\\ubuntu-llm\\home\\efcardoso\\projects\\b3-data-plataform\\i_notebooks\\..\\d_processing\\silver\\transform.py", "filename": "transform.py", "exc_info": null, "exc_text": null, "stack_info": null, "lineno": 159, "funcName": "write_silver", "created": 1782406919.3795233, "msecs": 379.0, "relativeCreated": 715690.6322, "thread": 8944, "threadName": "MainThread", "processName": "MainProcess", "process": 14144, "taskName": "Task-301", "rows": 720, "partitions": 60, "output": "\\\\wsl.localhost\\ubuntu-llm\\home\\efcardoso\\projects\\b3-data-plataform\\j_data\\silver"}
{"timestamp": "2026-06-25T17:02:02.736401+00:00", "level": "INFO", "log

ticker,trade_date,open_price,high_price,low_price,close_price,adj_close,volume,daily_return,processed_at
str,date,f64,f64,f64,f64,f64,i64,f64,"datetime[μs, UTC]"
"""ABEV3.SA""",2026-03-27,14.79,15.0,14.74,14.76,14.719015,17884700,-0.679688,2026-06-25 17:01:55.792161 UTC
"""BBAS3.SA""",2026-03-27,22.99,22.99,22.52,22.66,22.500587,16957900,-0.029966,2026-06-25 17:01:55.792161 UTC
"""BBDC4.SA""",2026-03-27,18.709999,18.809999,18.440001,18.52,18.1793,20727200,-0.625025,2026-06-25 17:01:55.792161 UTC
"""BPAC11.SA""",2026-03-27,54.77,54.779999,53.130001,53.529999,53.529999,14128200,2.226642,2026-06-25 17:01:55.792161 UTC
"""ITUB4.SA""",2026-03-27,41.810001,41.810001,41.18,41.450001,41.026432,27910700,-0.502102,2026-06-25 17:01:55.792161 UTC
"""LREN3.SA""",2026-03-27,15.11,15.26,14.67,14.89,14.655182,25661200,-0.691846,2026-06-25 17:01:55.792161 UTC
"""MGLU3.SA""",2026-03-27,8.41,8.45,8.08,8.09,8.01463,22237700,-0.833127,2026-06-25 17:01:55.792161 UTC
"""PETR4.SA""",2026-03-27,48.5,49.439999,48.130001,49.41,47.916618,54170900,0.145606,2026-06-25 17:01:55.792161 UTC
"""RADL3.SA""",2026-03-27,23.549999,23.66,23.1,23.299999,23.208319,4994500,-0.500107,2026-06-25 17:01:55.792161 UTC


## 5. Visualize daily returns

In [ ]:
import plotly.express as px
import plotly.io as pio

pio.templates.default = "plotly_white"

top5 = ["PETR4.SA", "VALE3.SA", "ITUB4.SA", "BBDC4.SA", "ABEV3.SA"]
plot_df = silver_df.filter(pl.col("ticker").is_in(top5)).sort("trade_date").to_pandas()

fig = px.line(
    plot_df,
    x="trade_date",
    y="daily_return",
    color="ticker",
    title="Retorno Diário das Ações da B3 — Camada Silver",
    labels={
        "trade_date": "Data do Pregão",
        "daily_return": "Retorno Diário",
        "ticker": "Ativo",
    },
)
fig.add_hline(y=0, line_dash="dash", line_color="grey", annotation_text="0%", annotation_position="top left")
fig.update_layout(
    height=500,
    hovermode="x unified",
    legend=dict(title="Ativo", orientation="v", x=1.02, y=1),
    margin=dict(l=60, r=40, t=70, b=50),
)
fig.update_yaxes(tickformat=".2%")
fig.update_xaxes(tickformat="%d/%m/%Y")
fig.show()

In [ ]:
# Distribuição dos retornos diários
fig2 = px.histogram(
    plot_df.dropna(subset=["daily_return"]),
    x="daily_return",
    color="ticker",
    nbins=60,
    title="Distribuição dos Retornos Diários por Ativo",
    barmode="overlay",
    opacity=0.55,
    labels={
        "daily_return": "Retorno Diário",
        "ticker": "Ativo",
        "count": "Frequência",
    },
)
fig2.update_layout(
    height=480,
    legend=dict(title="Ativo"),
    margin=dict(l=60, r=40, t=70, b=50),
    yaxis_title="Frequência",
)
fig2.update_xaxes(tickformat=".1%")
fig2.show()

## 6. Run full Silver pipeline

In [20]:
from f_pipelines.silver_pipeline import SilverPipeline

result = SilverPipeline().run()
print(f"SilverPipeline produced {len(result):,} clean rows")

{"timestamp": "2026-06-25T17:02:26.879128+00:00", "level": "INFO", "logger": "f_pipelines.silver_pipeline", "message": "SilverPipeline started", "module": "silver_pipeline", "func": "run", "line": 53, "name": "f_pipelines.silver_pipeline", "msg": "SilverPipeline started", "args": [], "levelname": "INFO", "levelno": 20, "pathname": "\\\\wsl.localhost\\ubuntu-llm\\home\\efcardoso\\projects\\b3-data-plataform\\i_notebooks\\..\\f_pipelines\\silver_pipeline.py", "filename": "silver_pipeline.py", "exc_info": null, "exc_text": null, "stack_info": null, "lineno": 53, "funcName": "run", "created": 1782406946.8790884, "msecs": 879.0, "relativeCreated": 743190.1974, "thread": 8944, "threadName": "MainThread", "processName": "MainProcess", "process": 14144, "taskName": "Task-328"}
{"timestamp": "2026-06-25T17:02:26.880040+00:00", "level": "INFO", "logger": "f_pipelines.silver_pipeline", "message": "Reading Bronze data", "module": "silver_pipeline", "func": "extract", "line": 39, "name": "f_pipelin